In [ ]:
Http
url - uniform resource locator(jsonplaceholder.typicode.com/users)
method (verb) - get/post/put/delete(CRUD)
headers - Metadata(application/json)
body(payload) - data being sent (post/put)

Client (requests)-> Server(response)
pending -> fulfilled -> rejected
status code 

200 OK - success response
201 Created - Success (post )
204 No Content - Success (there is nothing to return)
400 Bad Request: You sent invalid JSON or missing parameters.
401 Unauthorized: You are missing an API Key or Password.
403 Forbidden: You have a key, but you don't have permission for this specific file.
404 Not Found: The Endpoint (URL) is wrong or the ID doesn't exist.
429 Too Many Requests: You are spamming the server (Rate Limiting).
500 Internal Server Error: The server crashed. This is not your fault.


| Feature | When to use it | Why it's "Advanced" |
| :--- | :--- | :--- |
| **Named Groups** | Complex patterns | Prevents code from breaking if you add new groups. |
| **re.DOTALL** | Multi-line blocks | Makes the `.` match newline characters. |
| **Backreferences** | Data Reformatting | Lets you "shuffle" parts of a string around. |
| **re.finditer** | Large log files | Keeps RAM usage low by processing one match at a time. |
| **re.compile** | Regex inside loops | Speeds up execution by pre-calculating the pattern. |



In [ ]:
## 1. Organization: Named Groups
# Instead of counting parentheses, we give our data labels. Think of this as moving from an index-based list to a dictionary.

import re

log_entry = "Event: LOGIN_SUCCESS User: admin_01 Time: 14:30"

# THE PROFESSIONAL WAY: Named Groups
# Syntax: (?P<name>pattern) creates a group that can be referenced by 'name'

# Breaking down the regex pattern used below:
# r"..."                : Python raw string. It treats backslashes (\) as literal characters, saving us from writing double backslashes (\\w).
# Event:                : Matches the exact text "Event: "
# (?P<event>\w+)        : Named group 'event'. 
#                         - \w+ matches 1 or more "word characters" (letters, numbers, or underscores). This grabs "LOGIN_SUCCESS".
#  User:                : Matches the exact text " User: "
# (?P<user>\w+)         : Named group 'user'. 
#                         - \w+ again grabs 1 or more word characters. This grabs "admin_01".
#  Time:                : Matches the exact text " Time: "
# (?P<time>\d+:\d+)     : Named group 'time'. 
#                         - \d+ matches 1 or more digits.
#                         - ':' matches the exact colon character.
#                         - \d+ matches 1 or more digits.
#                         - Combined, \d+:\d+ cleanly matches our time format like "14:30".

robust_pattern = r"Event: (?P<event>\w+) User: (?P<user>\w+) Time: (?P<time>\d+:\d+)"

match = re.search(robust_pattern, log_entry)

if match:
    # Access by the labels we created
    print(f"--- Named Group Access ---")
    print(f"User: {match.group('user')}")
    print(f"Action: {match.group('event')}\n")
    
    # Powerful Feature: Convert entire match to a Dictionary
    data_dict = match.groupdict()
    print(f"As Dictionary: {data_dict}")


--- Named Group Access ---
User: admin_01
Action: LOGIN_SUCCESS

As Dictionary: {'event': 'LOGIN_SUCCESS', 'user': 'admin_01', 'time': '14:30'}


In [ ]:
# This code snippet demonstrates two very powerful features of Python's regular expressions: **Flags** 
# (which modify how the regex engine behaves)
# and **Substitution with Backreferences** (which allows you to reformat data based on patterns).


# Here is a detailed, step-by-step explanation of what is happening in both parts.

### Part A: Using Flags (`IGNORECASE` and `DOTALL`)

# Real-world strings are often messy involving weird capitalization or line breaks. Flags help you handle this without writing overly 
# complex regex patterns.

text = """
Start
Title: PYTHON 
Description: Python is a 
versatile language.
End
"""

pattern = r"title: (.*?) end"

# In the pattern above, you are looking for everything between `"title: "` and `" end"`. However, if you apply this pattern normally to the `text`, 
# it will fail for two reasons:
# 1. **Case sensitivity:** Your pattern is lowercase (`title: ` and ` end`), but the text has uppercase/mixed case (`Title: ` and `End`).
# 2. **Line breaks:** By default, the wild card character `.` matches any character *except* a newline (`\n`). Since the text between 
# "Title" and "End" spans multiple lines, the dot won't cross those line breaks.

# To fix this, the code uses two flags combined using the bitwise OR operator (`|`):


match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)

# *   **`re.IGNORECASE` (or `re.I`)**: Tells the regex engine to ignore capitalization. `title:` will successfully match `Title:`.
# *   **`re.DOTALL` (or `re.S`)**: Tells the regex engine to make the `.` character match *everything*, **including newlines**. 
# This allows the `(.*?)` to span across multiple lines and capture the entire description until it hits `End`.

# *(Note: `(.*?)` is a non-greedy wildcard, meaning it captures as little text as possible until it reaches the next part of the pattern).*


if match:
    print(match.group(1).strip()) 
# Once found, `match.group(1)` retrieves only the text captured within the parentheses `(.*?)`. `.strip()` is then used to clean up the 
# extra spaces and 
# line breaks at the beginning and end of the extracted string.


### Part B: Advanced Cleaning (Backreferences)

# This section shows how to find specific patterns and rearrange their internal components using **capture groups** and **backreferences**. 

dates = "Dates: 12/31/2023 and 01/15/2024"
date_pattern = r"(\d{2})/(\d{2})/(\d{4})"

# Here, the goal is to convert the US date format (`MM/DD/YYYY`) to the standard ISO format (`YYYY-MM-DD`). 
# The `pattern` puts parentheses around the different parts of the date to create **Capture Groups**:
# *   Group 1: `(\d{2})` captures the 2-digit **Month** (e.g., "12").
# *   Group 2: `(\d{2})` captures the 2-digit **Day** (e.g., "31").
# *   Group 3: `(\d{4})` captures the 4-digit **Year** (e.g., "2023").

# Once captured, you can refer back to these specific groups in your replacement string using **Backreferences** (`\1`, `\2`, `\3`).

reformatted = re.sub(date_pattern, r"\3-\1-\2", dates)

# *   **`re.sub(pattern, replacement, text)`**: This function finds every match of the pattern in the string and replaces it.
# *   **`r"\3-\1-\2"`**: This is the replacement string. Because it is an 'r-string' (raw string), it interprets `\3` not as an escape character,
# but as a command meaning *"insert whatever was found in Group 3 here"*. 
#     *   It places Group 3 (Year), followed by a hyphen `-`
#     *   Then Group 1 (Month), followed by a hyphen `-`
#     *   Then Group 2 (Day)
    
# Ultimately, `12/31/2023` (Group 1 / Group 2 / Group 3) becomes `2023-12-31` (Group 3 - Group 1 - Group 2). The script successfully reformats 
# all the dates sequentially in a single line of code!

In [ ]:
## 3. Optimization: finditer & Compile
# When dealing with "Big Data," we must be careful with memory. `finditer` processes matches one-by-one rather than loading them all at once.

# 1. Compilation: Do the "heavy lifting" of reading the pattern once
# Great for loops!
email_regex = re.compile(r"[\w.]+@[\w.]+\.\w+")

# 2. finditer: The memory-safe way to handle large files
massive_data = "Contact us at support@tech.com or sales@tech.com " * 1000

print("\n--- Efficiency Check ---")
count = 0
# finditer creates a generator, NOT a giant list
for match in email_regex.finditer(massive_data):
    # Process one match at a time
    count += 1

print(f"Processed {count} emails without crashing the system.")

In [2]:
import re

# 1. PRE-COMPILE: The "Robot" knows the instructions before opening the door.
# This pattern looks for "ERROR:" followed by a code and a message.
log_pattern = re.compile(r"ERROR:\s*(?P<code>\d+)\s*-\s*(?P<message>.*)")

# 2. OPEN THE FILE: We use a context manager ('with' statement)
# This opens the file safely without loading the whole thing into RAM.
try:
    with open("massive_server_log.txt", "r") as file:
        print("--- Starting Efficient Scan ---")
        
        # We read the file line by line to be memory-efficient
        for line_number, line in enumerate(file, 1):
            
            # 3. FINDITER: Find matches on THIS specific line
            for match in log_pattern.finditer(line):
                # Access data by the Names we gave in the pattern
                error_code = match.group('code')
                message = match.group('message')
                
                print(f"Line {line_number} | Code {error_code}: {message}")

except FileNotFoundError:
    print("File not found. Try creating a dummy .txt file to test!")

File not found. Try creating a dummy .txt file to test!


In [3]:
import random

# Create a fake log file with 10,000 entries
with open("vitals_log.txt", "w") as f:
    actions = ["LOGIN", "LOGOUT", "UPDATE", "DELETE"]
    statuses = ["INFO", "WARNING", "ERROR"]
    
    for i in range(10000):
        status = random.choice(statuses)
        # We'll make Errors look specific: ERROR: 500 - Database Timeout
        if status == "ERROR":
            line = f"L-{i} ERROR: {random.randint(400, 500)} - Connection Failure\n"
        else:
            line = f"L-{i} {status}: User_{random.randint(1, 99)} performed {random.choice(actions)}\n"
        f.write(line)

print("File 'vitals_log.txt' created successfully!")

File 'vitals_log.txt' created successfully!


In [4]:
import re

# 1. Compile once: (?P<code>\d+) captures the digits into a key named 'code'
error_pattern = re.compile(r"ERROR:\s*(?P<code>\d+)\s*-\s*(?P<msg>.*)")

error_count = 0

print(f"{'LINE':<8} | {'CODE':<6} | {'MESSAGE'}")
print("-" * 40)

# 2. Open the file as a stream (Memory Efficient)
with open("vitals_log.txt", "r") as file:
    for line in file:
        # 3. Use finditer to scan the current line
        for match in error_pattern.finditer(line):
            # Accessing data by name makes the code self-explanatory
            code = match.group('code')
            message = match.group('msg')
            
            # Print the first 5 errors found to show it works
            if error_count < 5:
                print(f"Match found! Code: {code} | Msg: {message}")
            
            error_count += 1

print("-" * 40)
print(f"Scan Complete. Total Errors Filtered: {error_count}")

LINE     | CODE   | MESSAGE
----------------------------------------
Match found! Code: 443 | Msg: Connection Failure
Match found! Code: 408 | Msg: Connection Failure
Match found! Code: 443 | Msg: Connection Failure
Match found! Code: 464 | Msg: Connection Failure
Match found! Code: 466 | Msg: Connection Failure
----------------------------------------
Scan Complete. Total Errors Filtered: 3284


In [8]:
from bs4 import BeautifulSoup

html_doc = """
<div id="main-container">
    <h1 id="title">Welcome to the Bookstore</h1>
    <div class="book-list">
        <article class="book special-offer">
            <h2 class="book-title">Python 101</h2>
            <div class="author">By <span class="name">Guido</span></div>
            <div class="meta">
                <span class="price">$29.99</span>
                <button class="btn buy-now" data-id="101">Add to Cart</button>
            </div>
        </article>
        <article class="book">
            <h2 class="book-title">Web Scraping Pro</h2>
            <div class="author">By <span class="name">Alice</span></div>
            <div class="meta">
                <span class="price">$45.00</span>
                <button class="btn out-of-stock" data-id="102">Sold Out</button>
            </div>
        </article>
    </div>
    <div class="footer">
        <div class="contact">Contact Us</div>
        <div class="socials">
            <a href="twitter.com">Twitter</a>
            <a href="linkedin.com">LinkedIn</a>
        </div>
    </div>
</div>
"""
soup = BeautifulSoup(html_doc, "html.parser")

# 1. Class Selection: Get all prices
prices = soup.select(".price")
print(f"Prices found: {[p.text for p in prices]}")

# 2. ID Selection: Get the unique title
main_title = soup.select_one("#title")
print(f"Main Title: {main_title.text}")

# 3. Chained Classes: Find a 'book' that is ALSO a 'special-offer'
special = soup.select_one(".book.special-offer")
print(f"Special Deal: {special.select_one('.book-title').text}")

# Descendant: Find 'a' anywhere inside '.footer'
all_links = soup.select(".footer a") # Returns 2 links
print(f"Total footer links: {len(all_links)}")

# Direct Child: Find 'div' immediately inside '.footer'
# It will find 'Contact Us' but NOT the 'a' tags (they are inside 'socials')
direct_divs = soup.select(".footer > div")
print(f"Direct children of footer: {[d.text.strip() for d in direct_divs]}")

# Syntax: [attribute='value']
buy_button = soup.select_one("button[data-id='101']")
print(f"Button Text: {buy_button.text}") # Output: Add to Cart

# STEP 1: Select all "Book" containers
books = soup.select(".book")

# STEP 2: Loop through each container individually
for book in books:
    # STEP 3: Scope your search ONLY to this book
    # Note: we use book.select_one() NOT soup.select_one()
    title = book.select_one(".book-title").text
    price = book.select_one(".price").text
    author = book.select_one(".name").text
    
    print(f"Book: {title} | Author: {author} | Price: {price}")

Prices found: ['$29.99', '$45.00']
Main Title: Welcome to the Bookstore
Special Deal: Python 101
Total footer links: 2
Direct children of footer: ['Contact Us', 'Twitter\nLinkedIn']
Button Text: Add to Cart
Book: Python 101 | Author: Guido | Price: $29.99
Book: Web Scraping Pro | Author: Alice | Price: $45.00


In [1]:
import requests
from bs4 import BeautifulSoup
import time

# --- SECTION 1: CONFIGURATION ---
# We start at Page 3 to demonstrate moving to Page 4, 5, etc.
base_url = "http://books.toscrape.com/catalogue/"
current_url = "http://books.toscrape.com/catalogue/page-3.html"

print("--- Starting Professional Crawler ---")

# --- SECTION 2: THE CRAWL LOOP (WHILE) ---
# We use a 'while' loop because we don't know exactly when the "Next" buttons will stop.
while current_url:
    print(f"\nFetching: {current_url}")
    
    try:
        # --- SECTION 3: NETWORK RELIABILITY ---
        response = requests.get(current_url, timeout=10)
        
        # This checks for 404 (Not Found) or 500 (Server Error)
        # If it finds one, it jumps straight to the 'except' block below.
        response.raise_for_status() 
        
        soup = BeautifulSoup(response.content, "html.parser")
        
        # --- SECTION 4: SCOPED EXTRACTION ---
        # We find all book containers on the current page
        books = soup.select(".product_pod")
        
        for book in books:
            item = {}
            
            # --- SECTION 5: HANDLING MESSY DATA (TRY/EXCEPT) ---
            # Real-world data is often missing. We wrap risky extractions.
            try:
                # 1. Extract Title (The 'title' attribute often has the full, uncut name)
                title_tag = book.select_one("h3 a")
                item['title'] = title_tag['title'] if title_tag else "N/A"
                
                # 2. Extract Price (High risk of missing or formatted strangely)
                price_tag = book.select_one(".price_color")
                item['price'] = price_tag.get_text(strip=True) if price_tag else "£0.00"
                
                # 3. Extract Availability
                stock_tag = book.select_one(".instock.availability")
                item['stock'] = stock_tag.get_text(strip=True) if stock_tag else "Out of Stock"
                
                print(f" Saved: {item['title'][:30]}... | {item['price']}")
                
            except Exception as e:
                # If a specific book is broken, we log it and move to the next book
                print(f" Warning: Skipped a book due to error: {e}")
                continue

        # --- SECTION 6: PAGINATION LOGIC ---
        # Look for the "next" button to find the next URL
        next_button = soup.select_one("li.next a")
        
        if next_button:
            # The href is often relative (e.g., 'page-4.html')
            # We combine it with our base_url to get the full link
            next_page_path = next_button["href"]
            
            # Logic: If the path doesn't have 'catalogue/', we add it. 
            # Books to Scrape has slightly inconsistent internal links.
            if "catalogue/" in next_page_path:
                current_url = "http://books.toscrape.com/" + next_page_path
            else:
                current_url = base_url + next_page_path
            
            # CRITICAL: Be a polite "bot." Pause so you don't overwhelm the server.
            time.sleep(1) 
        else:
            print("\nReached the final page. Finishing...")
            current_url = None # This breaks the 'while' loop

    except requests.exceptions.HTTPError as err:
        print(f" STOPPING: Network Error (Status {response.status_code})")
        break
    except Exception as err:
        print(f" STOPPING: Unexpected Error: {err}")
        break

print("--- Crawl Complete ---")

--- Starting Professional Crawler ---

Fetching: http://books.toscrape.com/catalogue/page-3.html
 Saved: Slow States of Collapse: Poems... | £57.31
 Saved: Reasons to Stay Alive... | £26.41
 Saved: Private Paris (Private #10)... | £47.61
 Saved: #HigherSelfie: Wake Up Your Li... | £23.11
 Saved: Without Borders (Wanderlove #1... | £45.07
 Saved: When We Collided... | £31.77
 Saved: We Love You, Charlie Freeman... | £50.27
 Saved: Untitled Collection: Sabbath P... | £14.27
 Saved: Unseen City: The Majesty of Pi... | £44.18
 Saved: Unicorn Tracks... | £18.78
 Saved: Unbound: How Eight Technologie... | £25.52
 Saved: Tsubasa: WoRLD CHRoNiCLE 2 (Ts... | £16.28
 Saved: Throwing Rocks at the Google B... | £31.12
 Saved: This One Summer... | £19.49
 Saved: Thirst... | £17.27
 Saved: The Torch Is Passed: A Harding... | £19.09
 Saved: The Secret of Dreadwillow Cars... | £56.13
 Saved: The Pioneer Woman Cooks: Dinne... | £56.41
 Saved: The Past Never Ends... | £56.50
 Saved: The Natural History 

In [1]:
import requests
from bs4 import BeautifulSoup
import time
import csv

# --- SECTION 1: SETUP THE CSV FILE ---
# 'w' means write mode. newline='' prevents extra blank rows in Windows.
with open("books.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    
    # Write the Header Row first
    writer.writerow(["Title", "Price", "Stock Status"])

    # --- SECTION 2: CRAWLER CONFIG ---
    base_url = "http://books.toscrape.com/catalogue/"
    current_url = "http://books.toscrape.com/catalogue/page-1.html"

    while current_url:
        print(f"Scraping: {current_url}")
        
        try:
            response = requests.get(current_url, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, "html.parser")
            
            books = soup.select(".product_pod")
            
            for book in books:
                # --- SECTION 3: ROBUST EXTRACTION ---
                try:
                    title = book.select_one("h3 a")["title"]
                    price = book.select_one(".price_color").get_text(strip=True)
                    stock = book.select_one(".instock.availability").get_text(strip=True)
                    
                    # --- SECTION 4: SAVE TO CSV ---
                    # We write a list representing one row of data
                    writer.writerow([title, price, stock])
                    
                except Exception:
                    continue # Skip broken entries

            # --- SECTION 5: NEXT PAGE LOGIC ---
            next_button = soup.select_one("li.next a")
            if next_button:
                next_page_path = next_button["href"]
                # URL Join Logic
                if "catalogue/" in next_page_path:
                    current_url = "http://books.toscrape.com/" + next_page_path
                else:
                    current_url = base_url + next_page_path
                
                time.sleep(1) # Be polite
            else:
                current_url = None # Break the loop

        except Exception as e:
            print(f"Error occurred: {e}")
            break

print("\nSuccess! Your data is saved in 'book_data.csv'.")

Scraping: http://books.toscrape.com/catalogue/page-1.html
Scraping: http://books.toscrape.com/catalogue/page-2.html
Scraping: http://books.toscrape.com/catalogue/page-3.html
Scraping: http://books.toscrape.com/catalogue/page-4.html
Scraping: http://books.toscrape.com/catalogue/page-5.html
Scraping: http://books.toscrape.com/catalogue/page-6.html
Scraping: http://books.toscrape.com/catalogue/page-7.html
Scraping: http://books.toscrape.com/catalogue/page-8.html
Scraping: http://books.toscrape.com/catalogue/page-9.html
Scraping: http://books.toscrape.com/catalogue/page-10.html
Scraping: http://books.toscrape.com/catalogue/page-11.html
Scraping: http://books.toscrape.com/catalogue/page-12.html
Scraping: http://books.toscrape.com/catalogue/page-13.html
Scraping: http://books.toscrape.com/catalogue/page-14.html
Scraping: http://books.toscrape.com/catalogue/page-15.html
Scraping: http://books.toscrape.com/catalogue/page-16.html
Scraping: http://books.toscrape.com/catalogue/page-17.html
Scrapi

In [2]:
import requests
from bs4 import BeautifulSoup
import time
import csv



with open("catalogue.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Title", "Price", "Status"])
    base_Url = "http://books.toscrape.com/catalogue/"
    current_url = "http://books.toscrape.com/catalogue/page-3.html"

    while current_url:
        print(f"Scraping: {current_url}")
        try:
            response = requests.get(current_url, timeout=5)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, "html.parser")
            
            books = soup.select(".product_pod")
            for book in books:
                item = {}
                try:
                    title_tag = book.select_one("h3 a")
                    item['title'] = title_tag['title'] if title_tag else "N/A"
                    price_tag = book.select_one(".price_color")
                    item['price'] = price_tag.get_text(strip=True) if price_tag else "£0.00"
                    stock_tag = book.select_one(".instock.availability")
                    item['stock'] = stock_tag.get_text(strip=True) if stock_tag else "Out of Stock"
                    writer.writerow([item['title'], item['price'], item['stock']])                    # print(item)
                except Exception as e:
                    print("Skipped a book due to error: {e}")
                    continue
            next_button = soup.select_one("li.next a") 
            
            if next_button:
                next_page = next_button["href"]
                if "catalogue/" in next_page:
                    current_url = "http://books.toscrape.com" + next_page
                else:
                    current_url = base_Url + next_page
                time.sleep(1)
            else:
                print("\nReached the final page. Finishing scraping...")
                current_url = None                  
        except requests.exceptions.HTTPError as err:
            print(f"Network Error (Status {response.status_code})")
            break
        except Exception as err:
            print(f"An unexpected Error occurred: {err}")
            break
print("Scraping complete!")    
            

Scraping: http://books.toscrape.com/catalogue/page-3.html
Scraping: http://books.toscrape.com/catalogue/page-4.html
Scraping: http://books.toscrape.com/catalogue/page-5.html
Scraping: http://books.toscrape.com/catalogue/page-6.html
Scraping: http://books.toscrape.com/catalogue/page-7.html
Scraping: http://books.toscrape.com/catalogue/page-8.html
Scraping: http://books.toscrape.com/catalogue/page-9.html
Scraping: http://books.toscrape.com/catalogue/page-10.html
Scraping: http://books.toscrape.com/catalogue/page-11.html
Scraping: http://books.toscrape.com/catalogue/page-12.html
Scraping: http://books.toscrape.com/catalogue/page-13.html
Scraping: http://books.toscrape.com/catalogue/page-14.html
Scraping: http://books.toscrape.com/catalogue/page-15.html
Scraping: http://books.toscrape.com/catalogue/page-16.html
Scraping: http://books.toscrape.com/catalogue/page-17.html
Scraping: http://books.toscrape.com/catalogue/page-18.html
Scraping: http://books.toscrape.com/catalogue/page-19.html
Scra

In [12]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

# --- SECTION 1: THE DATA CONTAINER ---
# We use a list to hold 'rows' of data. Each row is a dictionary.
books_data = []

url = "http://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

# --- SECTION 2: THE EXTRACTION LOOP ---
cards = soup.select(".product_pod")

for card in cards:
    # A. Scrape the raw values
    title = card.select_one("h3 a")["title"]
    price_raw = card.select_one(".price_color").text
    stock = card.select_one(".availability").get_text(strip=True)
    
    # B. Create the 'Row' (Dictionary)
    # The Keys ('Title', 'Price', etc.) will become your Column Headers
    book_item = {
        "Title": title,
        "Price_Raw": price_raw,
        "Stock_Status": stock,
        "Source_URL": url
    }
    
    # C. Append the row to our main list
    books_data.append(book_item)

# --- SECTION 3: THE PANDAS TRANSFORMATION ---
# Convert our list of dictionaries into a "DataFrame" (A virtual Spreadsheet)
df = pd.DataFrame(books_data)

# --- SECTION 4: DATA CLEANING ---
# We can't do math on "£51.77". We must remove the '£' and convert to a number.
# .str.replace() handles the text, .astype(float) handles the math conversion.
df['Price_Numeric'] = df['Price_Raw'].str.replace('£', '').astype(float)

# Example Analysis:
avg_price = df['Price_Numeric'].mean()
print(f"Extraction Complete! Average Price of these books: £{avg_price:.2f}")

# --- SECTION 5: THE EXPORT ---
# index=False prevents Pandas from adding an extra column for row numbers (0, 1, 2...)
df.to_csv("professional_books.csv", index=False)

print("File 'professional_books.csv' is ready for Excel!")

Extraction Complete! Average Price of these books: £38.05
File 'professional_books.csv' is ready for Excel!


In [4]:
def common_items(set1, set2):
    return set1.intersection(set2)

set1 = {1, 2, 3, 4}
set2 = {3, 4, 5, 6}

print(common_items(set1, set2))

{3, 4}


In [13]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tabulate import tabulate  # New: For the 'Good Table' look

# --- SECTION 1: THE DATA CONTAINER ---
books_data = []
url = "http://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

# --- SECTION 2: THE EXTRACTION LOOP ---
cards = soup.select(".product_pod")

for card in cards:
    title = card.select_one("h3 a")["title"]
    price_raw = card.select_one(".price_color").text
    stock = card.select_one(".availability").get_text(strip=True)
    
    book_item = {
        "Title": title,
        "Price_Raw": price_raw,
        "Stock_Status": stock
    }
    books_data.append(book_item)

# --- SECTION 3: THE PANDAS TRANSFORMATION ---
df = pd.DataFrame(books_data)

# --- SECTION 4: DATA CLEANING ---
df['Price_Num'] = df['Price_Raw'].str.replace('£', '').astype(float)

# --- SECTION 5: DISPLAYING THE "GOOD TABLE" ---
# We use 'tabulate' to print the DataFrame as a beautiful grid.
print("\n" + "="*30)
print(" ALL SCRAPED BOOKS ")
print("="*30)
print(tabulate(df, headers='keys', tablefmt='grid', showindex=False))

# --- SECTION 6: FILTERING THE TABLE ---
# Students often want to see specific data (e.g., only "In Stock" books)
print("\n" + "="*30)
print(" FILTERED: IN STOCK ONLY ")
print("="*30)

# Create a filter
in_stock_df = df[df['Stock_Status'] == "In stock"]

# Display the filtered version
print(tabulate(in_stock_df, headers='keys', tablefmt='fancy_grid', showindex=False))

# Final Metric
print(f"\nAverage Price: £{df['Price_Num'].mean():.2f}")


 ALL SCRAPED BOOKS 
+------------------------------------------------------------------------------------------------+-------------+----------------+-------------+
| Title                                                                                          | Price_Raw   | Stock_Status   |   Price_Num |
+================================================================================================+=============+================+=============+
| A Light in the Attic                                                                           | £51.77      | In stock       |       51.77 |
+------------------------------------------------------------------------------------------------+-------------+----------------+-------------+
| Tipping the Velvet                                                                             | £53.74      | In stock       |       53.74 |
+------------------------------------------------------------------------------------------------+-------------+---

In [15]:
# The Data: Representing a Library Catalog
# Notice the Root (catalog), Children (book), and Attributes (id, category)
xml_content = """<?xml version="1.0" encoding="UTF-8"?>
<catalog>
    <book id="bk101" category="Programming">
        <author>Gaddis, Tony</author>
        <title>Starting Out with Python</title>
        <price>89.95</price>
        <description>A comprehensive guide &amp; introduction to Python.</description>
    </book>
    <book id="bk102" category="Data Science">
        <author>McKinney, Wes</author>
        <title>Python for Data Analysis</title>
        <price>39.99</price>
        <description>The ultimate guide to Pandas &amp; Numpy.</description>
    </book>
</catalog>"""

print("XML string defined successfully.")

XML string defined successfully.


In [16]:
import xml.etree.ElementTree as ET

# Convert string to XML Tree
root = ET.fromstring(xml_content)

print(f"Root Tag: {root.tag}")
print("-" * 20)

# Traverse the tree
for book in root.findall('book'):
    # .get() extracts Attributes
    book_id = book.get('id')
    category = book.get('category')
    
    # .find().text extracts the content between tags
    title = book.find('title').text
    author = book.find('author').text
    price = book.find('price').text
    
    print(f"[{book_id}] {title}")
    print(f"Author: {author} | Category: {category}")
    print(f"Price: ${price}")
    print("-" * 20)

Root Tag: catalog
--------------------
[bk101] Starting Out with Python
Author: Gaddis, Tony | Category: Programming
Price: $89.95
--------------------
[bk102] Python for Data Analysis
Author: McKinney, Wes | Category: Data Science
Price: $39.99
--------------------


In [17]:
import pandas as pd
from tabulate import tabulate

rows = []

for book in root.findall('book'):
    rows.append({
        "ID": book.get('id'),
        "Title": book.find('title').text,
        "Author": book.find('author').text,
        "Price": float(book.find('price').text),
        "Category": book.get('category')
    })

# Create DataFrame
df = pd.DataFrame(rows)

# Visual Table Output
print(tabulate(df, headers='keys', tablefmt='fancy_grid', showindex=False))

╒═══════╤══════════════════════════╤═══════════════╤═════════╤══════════════╕
│ ID    │ Title                    │ Author        │   Price │ Category     │
╞═══════╪══════════════════════════╪═══════════════╪═════════╪══════════════╡
│ bk101 │ Starting Out with Python │ Gaddis, Tony  │   89.95 │ Programming  │
├───────┼──────────────────────────┼───────────────┼─────────┼──────────────┤
│ bk102 │ Python for Data Analysis │ McKinney, Wes │   39.99 │ Data Science │
╘═══════╧══════════════════════════╧═══════════════╧═════════╧══════════════╛


In [ ]:
from lxml import etree
import os

# --- STEP 1: Create the XSD File (The Blueprint) ---
xsd_content = """<?xml version="1.0" encoding="UTF-8"?>
<xs:schema xmlns:xs="http://www.w3.org/2001/XMLSchema">
  <xs:element name="inventory">
    <xs:complexType>
      <xs:sequence>
        <xs:element name="book" maxOccurs="unbounded">
          <xs:complexType>
            <xs:sequence>
              <xs:element name="title" type="xs:string"/>
              <xs:element name="pages" type="xs:integer"/>
              <xs:element name="price">
                <xs:complexType>
                  <xs:simpleContent>
                    <xs:extension base="xs:decimal">
                      <xs:attribute name="currency" type="xs:string" use="required"/>
                    </xs:extension>
                  </xs:simpleContent>
                </xs:complexType>
              </xs:element>
            </xs:sequence>
          </xs:complexType>
        </xs:element>
      </xs:sequence>
    </xs:complexType>
  </xs:element>
</xs:schema>"""

with open("inventory.xsd", "w") as f:
    f.write(xsd_content)

# --- STEP 2: Create a Valid XML File (The Data) ---
xml_content = """<?xml version="1.0" encoding="UTF-8"?>
<inventory>
    <book>
        <title>Python Mastery</title>
        <pages>350</pages>
        <price currency="USD">29.99</price>
    </book>
    <book>
        <title>XML Blueprints</title>
        <pages>120</pages>
        <price currency="EUR">15.50</price>
    </book>
</inventory>"""

with open("inventory.xml", "w") as f:
    f.write(xml_content)

# --- STEP 3: The Validation Function ---
def check_validity(xml_file, xsd_file):
    try:
        schema_root = etree.parse(xsd_file)
        schema = etree.XMLSchema(schema_root)
        xml_doc = etree.parse(xml_file)

        if schema.validate(xml_doc):
            print("Success: The XML follows all XSD rules!")
        else:
            print("Error: Validation Failed!")
            for error in schema.error_log:
                print(f"   Line {error.line}: {error.message}")
    except Exception as e:
        print(f"System Error: {e}")

# --- STEP 4: TRIGGER THE FUNCTION ---
check_validity("inventory.xml", "inventory.xsd")

✅ Success: The XML follows all XSD rules!


In [ ]:
from lxml import etree

def check_validity(xml_file, xsd_file):
    try:
        # 1. Load the Blueprint (XSD)
        schema_root = etree.parse(xsd_file)
        schema = etree.XMLSchema(schema_root)

        # 2. Load the Data (XML)
        xml_doc = etree.parse(xml_file)

        # 3. Validate
        if schema.validate(xml_doc):
            print("Success: XML matches the Blueprint.")
        else:
            print("Error: Invalid XML Structure.")
            # Print specific line numbers where the error occurred
            for log in schema.error_log:
                print(f"   Line {log.line}: {log.message}")

    except Exception as e:
        print(f"System Error: {e}")
        
# --- ADD THIS TO THE BOTTOM ---

# 1. Define your filenames (make sure these files exist in your folder!)
my_xml = "inventory.xml"
my_xsd = "inventory.xsd"

# 2. Call the function to actually run the logic
check_validity(my_xml, my_xsd)        

✅ Success: XML matches the Blueprint.


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

c:\Users\Admin\Desktop\python_foundations\api_test\myenv\Lib\site-packages\IPython\core\interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [24]:
from fastapi import FastAPI, Query, Path, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional

# 1. Global Documentation Metadata
app = FastAPI(
    title="Titan Logistics API",
    description="""
    This API manages real-time drone deliveries.
    
    ## Features
    * **Track** packages in real-time.
    * **Schedule** new pick-ups.
    * **Manage** fleet status.
    """,
    version="2.1.0",
    contact={
        "name": "Dev Support",
        "url": "https://support.titan.com",
    }
)

# 2. Documenting the Data Model (The "Schema")
class Delivery(BaseModel):
    id: int = Field(..., example=101, description="Unique identifier for the delivery")
    package_name: str = Field(..., example="Medical Supplies", min_length=3)
    weight_kg: float = Field(..., gt=0, le=50, example=12.5)
    status: str = Field(
        default="Pending", 
        description="Current state of delivery",
        pattern="^(Pending|In-Flight|Delivered)$"
    )

# 3. Documenting the Endpoint with Tags and Summaries
@app.get(
    "/deliveries/{delivery_id}",
    response_model=Delivery,
    tags=["Fleet Operations"],
    summary="Get status of a specific drone",
    response_description="Returns the current GPS and battery status of the drone."
)
async def get_delivery(
    delivery_id: int = Path(..., description="The ID of the delivery to look up", gt=0)
):
    """
    This description will show up as the **detailed long-form text** inside the documentation UI.
    """
    # Logic to fetch from DB would go here
    return {"id": delivery_id, "package_name": "Medical Supplies", "weight_kg": 12.5, "status": "In-Flight"}

C:\Users\Admin\AppData\Local\Temp\ipykernel_22128\1077755552.py:25: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  id: int = Field(..., example=101, description="Unique identifier for the delivery")
C:\Users\Admin\AppData\Local\Temp\ipykernel_22128\1077755552.py:26: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  package_name: str = Field(..., example="Medical Supplies", min_length=3)
C:\Users\Admin\AppData\Local\Temp\ipykernel_22128\1077755552.py:27: PydanticDeprecatedSince20: Using extra keyword argume

In [4]:
import requests

url = "https://jsonplaceholder.typicode.com/users/1"

response = requests.get(url)

print(f"The api responded with: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    print(data['name'])
    print(data['username'])
    print(data['address'])

The api responded with: 200
Leanne Graham
Bret
{'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}


In [5]:
import json
import requests

url = "https://jsonplaceholder.typicode.com/users/1"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    
    with open("user_data.json", "w") as file:
        # indent=4 makes the file look pretty instead of one long line
        json.dump(data, file, indent=4)
        
    print("Full JSON object saved.")

Full JSON object saved.


In [2]:
import requests

url = "https://jsonplaceholder.typicode.com/users/1"
response = requests.get(url)

print(f"The api responded with: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    
    # Open file in 'w' mode (this creates the file or overwrites it)
    with open("user_data.txt", "w") as file:
        file.write(f"Name: {data['name']}\n")
        file.write(f"Username: {data['username']}\n")
        
        # Since 'address' is a dictionary, we format it nicely
        addr = data['address']
        file.write(f"Address: {addr['street']}, {addr['city']}\n")
        
    print("Data successfully written to user_data.txt")

The api responded with: 200
Data successfully written to user_data.txt


In [7]:
import requests

base_Url = 'https://jsonplaceholder.typicode.com/posts'
search_terms = {
    'userId': 1,
    'completed': 'false'
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36...",
    "Accept": "application/json"
}

response = requests.get(base_Url, params=search_terms, headers=headers)

data = response.json()

print(response.url)

https://jsonplaceholder.typicode.com/posts?userId=1&completed=false
